In [0]:
# Databricks notebook: 08_aggregated_merchants
# Purpose: Transform PhonePe aggregated merchant JSON into
# an analytics-ready Silver Delta table.

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    LongType
)

# ============================================================
# 1. CONFIGURATION
# ============================================================

BRONZE_TABLE = "workspace.bronze.phonepe_raw_json"
METADATA_TABLE = "workspace.silver.phonepe_file_metadata"
SILVER_TABLE = "workspace.silver.aggregated_merchants"

# 34 country files + 1,178 available state files
EXPECTED_SOURCE_FILES = 1_212
EXPECTED_COUNTRY_FILES = 34
EXPECTED_STATE_FILES = 1_178

print("Configuration")
print(f"Bronze source : {BRONZE_TABLE}")
print(f"Metadata source: {METADATA_TABLE}")
print(f"Silver target : {SILVER_TABLE}")
print(f"Expected files: {EXPECTED_SOURCE_FILES:,}")


# ============================================================
# 2. SELECT AGGREGATED MERCHANT FILES
# ============================================================

bronze_df = spark.table(BRONZE_TABLE).alias("b")

merchant_metadata_df = (
    spark.table(METADATA_TABLE)
    .filter(
        (F.col("dataset_group") == "aggregated") &
        (F.col("domain") == "merchant")
    )
    .alias("m")
)

merchant_source_df = (
    merchant_metadata_df
    .join(
        bronze_df,
        F.col("m.source_path") == F.col("b.source_path"),
        "inner"
    )
    .select(
        F.col("m.relative_path"),
        F.col("m.file_scope"),
        F.col("m.state_slug"),
        F.col("m.year"),
        F.col("m.quarter"),
        F.col("m.period"),
        F.col("b.raw_json"),
        F.col("b.source_path"),
        F.col("b.source_commit"),
        F.col("b.content_sha256"),
        F.col("b.ingested_at_utc")
    )
)

source_scope_summary = (
    merchant_source_df
    .groupBy("file_scope")
    .agg(
        F.countDistinct("source_path").alias("file_count")
    )
)

source_counts = {
    row["file_scope"]: row["file_count"]
    for row in source_scope_summary.collect()
}

actual_country_files = source_counts.get("country", 0)
actual_state_files = source_counts.get("state", 0)
actual_source_files = actual_country_files + actual_state_files

print("\nSource validation")
print(f"Country files: {actual_country_files:,}")
print(f"State files  : {actual_state_files:,}")
print(f"Total files  : {actual_source_files:,}")

assert actual_country_files == EXPECTED_COUNTRY_FILES, (
    f"Expected {EXPECTED_COUNTRY_FILES:,} country files, "
    f"found {actual_country_files:,}."
)

assert actual_state_files == EXPECTED_STATE_FILES, (
    f"Expected {EXPECTED_STATE_FILES:,} state files, "
    f"found {actual_state_files:,}."
)

assert actual_source_files == EXPECTED_SOURCE_FILES

print("Aggregated merchant source validation passed.")


# ============================================================
# 3. DEFINE EXPLICIT JSON SCHEMA
# ============================================================

aggregated_merchant_schema = StructType([
    StructField("success", BooleanType(), True),
    StructField("code", StringType(), True),

    StructField(
        "data",
        StructType([
            StructField(
                "aggregated",
                StructType([
                    StructField(
                        "registeredCount",
                        LongType(),
                        True
                    )
                ]),
                True
            )
        ]),
        True
    ),

    StructField("responseTimestamp", LongType(), True)
])

print("\nExplicit aggregated merchant schema created.")


# ============================================================
# 4. PARSE JSON
# ============================================================

parsed_merchant_df = (
    merchant_source_df
    .withColumn(
        "parsed_json",
        F.from_json(
            F.col("raw_json"),
            aggregated_merchant_schema
        )
    )
)

parse_summary = (
    parsed_merchant_df
    .agg(
        F.count("*").alias("source_files"),

        F.sum(
            F.when(
                F.col(
                    "parsed_json.data.aggregated.registeredCount"
                ).isNull(),
                1
            ).otherwise(0)
        ).alias("files_without_registered_count")
    )
    .first()
)

print("\nJSON parsing")
print(f"Source files                  : {parse_summary['source_files']:,}")
print(
    f"Files without registered count: "
    f"{parse_summary['files_without_registered_count']:,}"
)

assert parse_summary["source_files"] == EXPECTED_SOURCE_FILES

assert parse_summary["files_without_registered_count"] == 0, (
    "One or more files do not contain a readable "
    "merchant registeredCount."
)

print("Aggregated merchant JSON parsing passed.")


# ============================================================
# 5. CREATE SILVER DATAFRAME
# ============================================================

aggregated_merchants_df = (
    parsed_merchant_df
    .select(
        F.col("file_scope"),
        F.col("state_slug"),
        F.col("year").cast("int").alias("year"),
        F.col("quarter").cast("int").alias("quarter"),
        F.col("period"),

        F.col(
            "parsed_json.data.aggregated.registeredCount"
        )
        .cast("long")
        .alias("registered_merchant_count"),

        F.col("parsed_json.responseTimestamp")
            .alias("response_timestamp_epoch_ms"),

        F.col("relative_path"),
        F.col("source_path"),
        F.col("source_commit"),
        F.col("content_sha256"),
        F.col("ingested_at_utc")
    )
)

print("\nAggregated merchant records transformed.")


# ============================================================
# 6. DATA-QUALITY VALIDATION
# ============================================================

quality_result = (
    aggregated_merchants_df
    .agg(
        F.count("*").alias("total_rows"),

        F.sum(
            F.when(
                F.col("registered_merchant_count").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_merchant_counts"),

        F.sum(
            F.when(
                F.col("registered_merchant_count") < 0,
                1
            ).otherwise(0)
        ).alias("negative_merchant_counts"),

        F.countDistinct("source_path")
            .alias("represented_source_files"),

        F.min("registered_merchant_count")
            .alias("minimum_merchant_count"),

        F.max("registered_merchant_count")
            .alias("maximum_merchant_count")
    )
    .first()
)

duplicate_business_keys = (
    aggregated_merchants_df
    .groupBy(
        "file_scope",
        "state_slug",
        "year",
        "quarter"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("\nData-quality results")
print(f"Total Silver rows        : {quality_result['total_rows']:,}")
print(
    f"Missing merchant counts  : "
    f"{quality_result['missing_merchant_counts']:,}"
)
print(
    f"Negative merchant counts : "
    f"{quality_result['negative_merchant_counts']:,}"
)
print(
    f"Source files represented : "
    f"{quality_result['represented_source_files']:,}"
)
print(
    f"Minimum merchant count   : "
    f"{quality_result['minimum_merchant_count']:,}"
)
print(
    f"Maximum merchant count   : "
    f"{quality_result['maximum_merchant_count']:,}"
)
print(f"Duplicate business keys  : {duplicate_business_keys:,}")

assert quality_result["total_rows"] == EXPECTED_SOURCE_FILES
assert quality_result["missing_merchant_counts"] == 0
assert quality_result["negative_merchant_counts"] == 0
assert quality_result["represented_source_files"] == EXPECTED_SOURCE_FILES
assert duplicate_business_keys == 0

print("Aggregated merchant quality checks passed.")


# ============================================================
# 7. SAVE MANAGED DELTA TABLE
# ============================================================

(
    aggregated_merchants_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"\nSilver Delta table created: {SILVER_TABLE}")


# ============================================================
# 8. VALIDATE SAVED TABLE
# ============================================================

saved_merchants_df = spark.table(SILVER_TABLE)

saved_summary = (
    saved_merchants_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("source_path").alias("source_file_count"),
        F.min("period").alias("first_period"),
        F.max("period").alias("last_period")
    )
    .first()
)

print("\nSaved-table validation")
print(f"Saved rows              : {saved_summary['row_count']:,}")
print(f"Source files represented: {saved_summary['source_file_count']:,}")
print(f"First period            : {saved_summary['first_period']}")
print(f"Last period             : {saved_summary['last_period']}")

assert saved_summary["row_count"] == EXPECTED_SOURCE_FILES
assert saved_summary["source_file_count"] == EXPECTED_SOURCE_FILES
assert saved_summary["first_period"] == "2018-Q1"
assert saved_summary["last_period"] == "2026-Q2"

print("Final aggregated merchant Delta-table validation passed.")


# ============================================================
# 9. DISPLAY SAMPLE
# ============================================================

display(
    saved_merchants_df
    .select(
        "file_scope",
        "state_slug",
        "year",
        "quarter",
        "period",
        "registered_merchant_count"
    )
    .orderBy(
        "year",
        "quarter",
        "file_scope",
        "state_slug"
    )
    .limit(20)
)

Configuration
Bronze source : workspace.bronze.phonepe_raw_json
Metadata source: workspace.silver.phonepe_file_metadata
Silver target : workspace.silver.aggregated_merchants
Expected files: 1,212

Source validation
Country files: 34
State files  : 1,178
Total files  : 1,212
Aggregated merchant source validation passed.

Explicit aggregated merchant schema created.

JSON parsing
Source files                  : 1,212
Files without registered count: 0
Aggregated merchant JSON parsing passed.

Aggregated merchant records transformed.

Data-quality results
Total Silver rows        : 1,212
Missing merchant counts  : 0
Negative merchant counts : 0
Source files represented : 1,212
Minimum merchant count   : 1
Maximum merchant count   : 50,705,249
Duplicate business keys  : 0
Aggregated merchant quality checks passed.

Silver Delta table created: workspace.silver.aggregated_merchants

Saved-table validation
Saved rows              : 1,212
Source files represented: 1,212
First period            

file_scope,state_slug,year,quarter,period,registered_merchant_count
country,null,2018,1,2018-Q1,13116
state,andhra-pradesh,2018,1,2018-Q1,30
state,bihar,2018,1,2018-Q1,2
state,chhattisgarh,2018,1,2018-Q1,10
state,delhi,2018,1,2018-Q1,35
state,gujarat,2018,1,2018-Q1,23
state,haryana,2018,1,2018-Q1,4
state,jammu-&-kashmir,2018,1,2018-Q1,2
state,karnataka,2018,1,2018-Q1,11162
state,kerala,2018,1,2018-Q1,3


# Silver Aggregated Merchants — Completion Summary

The `workspace.silver.aggregated_merchants` Delta table was successfully created and validated.

## Results

- Total records: **1,212**
- Country files: **34**
- Available state files: **1,178**
- Data coverage: **2018-Q1 to 2026-Q2**

## Table grain

Each record represents one country or state for one year and quarter.

## Available metric

- Cumulative registered-merchant count

## Analytical interpretation

`registered_merchant_count` is a cumulative stock metric. It must not be summed across quarters.

The Gold/dbt layer will derive:

- Quarterly merchant additions
- Merchant growth rates
- Transactions per merchant
- Transaction value per merchant
- Users per merchant
- Regional merchant-infrastructure density

## Source availability

The merchant state dataset contains fewer files than the transaction and user datasets. The pipeline validates against the available **1,178 state files** rather than assuming that every state-quarter combination exists.